# Instalação e Importação de Bibliotecas

In [0]:
%pip install databricks-feature-engineering
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, FunctionTransformer, MinMaxScaler
import mlflow
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# Amostragem


Esta etapa tem como objetivo **construir a base utilizada no treinamento e avaliação do modelo**, integrando as informações disponíveis no Feature Store e realizando uma divisão temporal dos dados.

Primeiramente, a query SQL responsável por selecionar os registros é carregada e executada. Em seguida, são utilizadas as tabelas do **Feature Store** para recuperar as diferentes categorias de variáveis utilizadas no modelo, como informações cadastrais, temporais, histórico financeiro, renda, funcionários e histórico de pagamentos.

O FeatureEngineeringClient realiza essa integração por meio dos FeatureLookup, utilizando ID_CLIENTE, ID_DOCUMENTO e DATA_REF como chaves de busca.

Após a criação do conjunto de treinamento, os dados são ordenados pela **data de referência (DATA_REF)** e divididos temporalmente:

| Conjunto  | Proporção |
| --------- | --------: |
| Treino    |       60% |
| Validação |       20% |
| Teste     |       20% |

A divisão temporal é utilizada para **simular um cenário real de crédito**, no qual o modelo é treinado utilizando informações do passado e posteriormente aplicado a períodos futuros.

Por fim, a variável FL_INAD é separada como **target**, enquanto as demais variáveis são utilizadas como features:

* X_train, X_val, X_test: variáveis preditoras;
* y_train, y_val, y_test: variável alvo de inadimplência.

Essa abordagem evita uma divisão aleatória dos dados e reduz o risco de **vazamento temporal**, proporcionando uma avaliação mais próxima das condições de utilização do modelo em produção.


In [0]:
# Função para importar query SQL de um arquivo
def import_query(path):
    with open(path) as f:
        return f.read()

# Carrega query SQL
query = import_query("fl_inad.sql")
df = spark.sql(query)

# Lista de FeatureLookups para buscar features no Feature Store
feature_lookups = [
    FeatureLookup(table_name="feature_store.credit_score.fs_cadastral", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_temporal", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_historico_financeiro", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_renda", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_funcionarios", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_historico_pagamentos", lookup_key=["ID_CLIENTE", "ID_DOCUMENTO", "DATA_REF"])
]

# Cria objeto FeatureEngineeringClient
fe = FeatureEngineeringClient()

# Cria o training_set com features e label
training_set = fe.create_training_set(df=df, feature_lookups=feature_lookups, label="FL_INAD")

# Exibe o dataframe do training_set
training_set.load_df().display()


In [0]:
df_train  = training_set.load_df().toPandas()

In [0]:
# Ordena o dataframe pelo tempo de referência
df_sorted = df_train.sort_values('DATA_REF').reset_index(drop=True)

# Define tamanhos dos conjuntos de treino, validação e teste
n = len(df_sorted)
n_train = int(0.6 * n)
n_val   = int(0.2 * n)
n_test  = n - n_train - n_val

# Split temporal: primeiro 60% para treino, 20% para validação, 20% para teste
train_df = df_sorted.iloc[:n_train]
val_df   = df_sorted.iloc[n_train:n_train + n_val]
test_df  = df_sorted.iloc[n_train + n_val:]

# Separa features e target
X_train, y_train = train_df.drop(columns=['FL_INAD'], errors='ignore'), train_df['FL_INAD']
X_val,   y_val   = val_df.drop(columns=['FL_INAD'], errors='ignore'),   val_df['FL_INAD']
X_test,  y_test  = test_df.drop(columns=['FL_INAD'], errors='ignore'),  test_df['FL_INAD']

# Exploração


Nesta etapa, é realizada uma análise inicial dos dados de treino para **entender o comportamento das variáveis, identificar valores ausentes e definir estratégias de tratamento**.

Primeiramente, as médias das variáveis são comparadas entre clientes adimplentes e inadimplentes, permitindo identificar diferenças relevantes entre os grupos. Também é calculado o percentual de valores nulos de cada variável.

Em seguida, é criado um NullImputer responsável pelo tratamento dos valores ausentes. A imputação é feita de acordo com o significado de cada variável, utilizando estratégias como **moda, média por porte, mediana, valores constantes e preenchimento em cascata**. Também são criadas flags indicando quando determinados valores foram originalmente ausentes, preservando essa informação para o modelo.

Por fim, são realizadas transformações nas variáveis, como **extração de informações das datas, transformação ordinal de PORTE e codificação de ESTADO e DDD**. Após essas etapas, são identificadas automaticamente as variáveis numéricas que serão utilizadas no restante do pré-processamento e modelagem.


In [0]:
df_train_explore = X_train.copy()
df_train_explore["target"] = y_train

describe = (
    df_train_explore
    .groupby("target")
    .mean(numeric_only=True)
    .T
)

describe["variable"] = describe.index
describe["ratio"] = describe[1] / describe[0].replace(0, np.nan)

cols = ["variable"] + [col for col in describe.columns if col != "variable"]
describe = describe[cols]

display(describe)

In [0]:
# Calcula a porcentagem de valores NaN por coluna
nan_pct = X_train.isna().mean() * 100

# Filtra apenas colunas com NaN e ordena decrescentemente
nan_pct = nan_pct[nan_pct > 0].sort_values(ascending=False)

# Converte para DataFrame para visualização
nan_pct_df = nan_pct.to_frame('Porcentagem_NaN').reset_index().rename(columns={'index': 'Coluna'})

display(nan_pct_df)

In [0]:
# Imputador de valores nulos
class NullImputer(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass

    def fit(self, X, y=None):
        X = X.copy()
        X["PORTE"] = X["PORTE"].fillna("DESCONHECIDO")
        X["DDD"] = pd.to_numeric(X["DDD"], errors="coerce")

        # Modas para imputação de categorias
        self.estado_moda_ = X["ESTADO"].mode()[0]
        self.cep_moda_ = X["CEP_2_DIG"].mode()[0]
        self.regiao_moda_ = X["REGIAO"].mode()[0]
        self.ddd_moda_ = X["DDD"].mode()[0]
        self.ddd_estado_ = (
            X.groupby("ESTADO")["DDD"]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
            .to_dict()
        )

        # Imputação por média de porte e global para variáveis de nível
        self.cols_nivel = [
            'MED_RENDA_3M', 'MIN_RENDA_3M', 'MAX_RENDA_3M', 'SOMA_RENDA_3M',
            'MED_RENDA_6M', 'MIN_RENDA_6M', 'MAX_RENDA_6M', 'SOMA_RENDA_6M',
            'MED_RENDA_1A', 'MIN_RENDA_1A', 'MAX_RENDA_1A', 'SOMA_RENDA_1A',
            'MED_RENDA_VIDA', 'MIN_RENDA_VIDA', 'MAX_RENDA_VIDA',
        ]
        self.media_porte_ = {}
        self.media_global_ = {}
        for col in self.cols_nivel:
            self.media_porte_[col] = X.groupby("PORTE")[col].mean().to_dict()
            self.media_global_[col] = X[col].mean()

        # Constantes para imputação
        self.fill_zero_ = [
            'DIAS_ANTECIPADO_MEDIA_3M',
            'DIAS_ANTECIPADO_MEDIA_6M',
            'DIAS_ANTECIPADO_MEDIA_12M',
            'DIAS_ANTECIPADO_MEDIA_VIDA',
            'CRESCIMENTO_FUNC_3M',
            'CRESCIMENTO_FUNC_6M',
            'CRESCIMENTO_FUNC_12M',
            'CRESCIMENTO_FUNC_VIDA',
            'FLAG_PORTE_AUSENTE',
            'FLAG_HISTORICO_3M',
            'FLAG_HISTORICO_6M',
            'FLAG_HISTORICO_12M',
            'QT_SAFRAS_HISTORICO',
            'FLAG_HISTORICO_VARIACAO',
            'MAIOR_CRESCIMENTO_MENSAL',
            'MAIOR_QUEDA_MENSAL',
        ]
        self.fill_9999_ = [
            'DIAS_DESDE_ULTIMO_ATRASO',
            'DIAS_DESDE_ULTIMA_INADIMPLENCIA',
            'DIAS_ULT_PAG',
        ]
        self.fill_menos1_ = [
            'NO_FUNCIONARIOS_ATUAL',
            'NO_FUNCIONARIOS_3M',
            'NO_FUNCIONARIOS_6M',
            'NO_FUNCIONARIOS_12M',
            'NO_FUNCIONARIOS_VIDA',
        ]

        # Variáveis de variação e std
        self.cols_variacao = [
            'CRESCIMENTO_ABS_RENDA_3M', 'CRESCIMENTO_PERC_RENDA_3M',
            'CRESCIMENTO_ABS_RENDA_6M', 'CRESCIMENTO_PERC_RENDA_6M',
            'CRESCIMENTO_ABS_RENDA_1A', 'CRESCIMENTO_PERC_RENDA_1A',
            'MIN_DIFF_VALOR_RENDA_3M', 'MED_DIFF_VALOR_RENDA_3M', 'MAX_DIFF_VALOR_RENDA_3M',
            'MIN_RAZAO_VALOR_RENDA_3M', 'MED_RAZAO_VALOR_RENDA_3M', 'MAX_RAZAO_VALOR_RENDA_3M',
            'MIN_DIFF_VALOR_RENDA_6M', 'MED_DIFF_VALOR_RENDA_6M', 'MAX_DIFF_VALOR_RENDA_6M',
            'MIN_RAZAO_VALOR_RENDA_6M', 'MED_RAZAO_VALOR_RENDA_6M', 'MAX_RAZAO_VALOR_RENDA_6M',
            'MIN_DIFF_VALOR_RENDA_1A', 'MED_DIFF_VALOR_RENDA_1A', 'MAX_DIFF_VALOR_RENDA_1A',
            'MIN_RAZAO_VALOR_RENDA_1A', 'MED_RAZAO_VALOR_RENDA_1A', 'MAX_RAZAO_VALOR_RENDA_1A',
            'MIN_DIFF_VALOR_RENDA_VIDA', 'MED_DIFF_VALOR_RENDA_VIDA', 'MAX_DIFF_VALOR_RENDA_VIDA',
            'MIN_RAZAO_VALOR_RENDA_VIDA', 'MED_RAZAO_VALOR_RENDA_VIDA', 'MAX_RAZAO_VALOR_RENDA_VIDA',
        ]
        self.cols_std = [
            'STD_RENDA_3M', 'STD_RENDA_6M', 'STD_RENDA_1A', 'STD_RENDA_VIDA',
        ]

        # Cascatas para imputação sequencial
        self.cascatas = {
            'DIAS_ATRASO_MEDIA':  ['DIAS_ATRASO_MEDIA_3M', 'DIAS_ATRASO_MEDIA_6M', 'DIAS_ATRASO_MEDIA_12M'],
            'DIAS_ATRASO_MIN':    ['DIAS_ATRASO_MIN_3M', 'DIAS_ATRASO_MIN_6M', 'DIAS_ATRASO_MIN_12M'],
            'DIAS_ATRASO_MAX':    ['DIAS_ATRASO_MAX_3M', 'DIAS_ATRASO_MAX_6M', 'DIAS_ATRASO_MAX_12M'],
            'DIAS_EMISSAO_PAGAMENTO_MEDIA': [
                'DIAS_EMISSAO_PAGAMENTO_MEDIA_3M',
                'DIAS_EMISSAO_PAGAMENTO_MEDIA_6M',
                'DIAS_EMISSAO_PAGAMENTO_MEDIA_12M'
            ],
            'VALOR_COBRANCA_DIA_MEDIA': [
                'VALOR_COBRANCA_DIA_MEDIA_3M',
                'VALOR_COBRANCA_DIA_MEDIA_6M',
                'VALOR_COBRANCA_DIA_MEDIA_12M',
                'VALOR_COBRANCA_DIA_MEDIA_VIDA'
            ],
            'VALOR_A_PAGAR_MIN': [
                'VALOR_A_PAGAR_MIN_3M',
                'VALOR_A_PAGAR_MIN_6M',
                'VALOR_A_PAGAR_MIN_12M',
                'VALOR_A_PAGAR_MIN_VIDA'
            ],
            'VALOR_A_PAGAR_MEDIA': [
                'VALOR_A_PAGAR_MEDIA_3M',
                'VALOR_A_PAGAR_MEDIA_6M',
                'VALOR_A_PAGAR_MEDIA_12M',
                'VALOR_A_PAGAR_MEDIA_VIDA'
            ],
            'VALOR_A_PAGAR_MAX': [
                'VALOR_A_PAGAR_MAX_3M',
                'VALOR_A_PAGAR_MAX_6M',
                'VALOR_A_PAGAR_MAX_12M',
                'VALOR_A_PAGAR_MAX_VIDA'
            ],
            'QT_ATRASO_MIN': ['QT_ATRASO_MIN_3M', 'QT_ATRASO_MIN_6M', 'QT_ATRASO_MIN_12M'],
            'QT_ATRASO_MEDIA': ['QT_ATRASO_MEDIA_3M', 'QT_ATRASO_MEDIA_6M', 'QT_ATRASO_MEDIA_12M'],
            'QT_ATRASO_MAX': ['QT_ATRASO_MAX_3M', 'QT_ATRASO_MAX_6M', 'QT_ATRASO_MAX_12M'],
            'PCT_EM_DIA': ['PCT_EM_DIA_3M', 'PCT_EM_DIA_6M', 'PCT_EM_DIA_12M'],
            'PCT_FORA_INADIMPLENCIA': ['PCT_FORA_INADIMPLENCIA_3M', 'PCT_FORA_INADIMPLENCIA_6M', 'PCT_FORA_INADIMPLENCIA_12M'],
            'PRAZO_EMISSAO_VENCIMENTO_MEDIA': [
                'PRAZO_EMISSAO_VENCIMENTO_MEDIA_3M',
                'PRAZO_EMISSAO_VENCIMENTO_MEDIA_6M',
                'PRAZO_EMISSAO_VENCIMENTO_MEDIA_12M'
            ],
            'PRAZO_EMISSAO_VENCIMENTO_MAX': [
                'PRAZO_EMISSAO_VENCIMENTO_MAX_3M',
                'PRAZO_EMISSAO_VENCIMENTO_MAX_6M',
                'PRAZO_EMISSAO_VENCIMENTO_MAX_12M'
            ],
            'PRAZO_EMISSAO_VENCIMENTO_MIN': [
                'PRAZO_EMISSAO_VENCIMENTO_MIN_3M',
                'PRAZO_EMISSAO_VENCIMENTO_MIN_6M',
                'PRAZO_EMISSAO_VENCIMENTO_MIN_12M'
            ],
        }
        self.medianas_cascata_ = {}
        for _, cols in self.cascatas.items():
            serie = X[cols].bfill(axis=1).iloc[:,0]
            med = serie.median()
            if pd.isna(med):
                med = 0
            self.medianas_cascata_[cols[0]] = med

        # Medianas para colunas específicas
        cols_restantes = [
            'VALOR_COBRANCA_DIA_MEDIA_6M',
            'VALOR_A_PAGAR_MEDIA_VIDA',
            'VALOR_A_PAGAR_MIN_VIDA',
            'VALOR_A_PAGAR_MAX_VIDA',
            'VALOR_A_PAGAR_MAX_6M',
            'VALOR_A_PAGAR_MIN_6M',
            'VALOR_A_PAGAR_MEDIA_6M',
            'DIAS_EMISSAO_PAGAMENTO_MEDIA_6M',
            'PRAZO_EMISSAO_VENCIMENTO_MEDIA_6M',
            'PRAZO_EMISSAO_VENCIMENTO_MIN_6M',
            'PCT_EM_DIA_6M',
            'PRAZO_EMISSAO_VENCIMENTO_MAX_6M',
            'PCT_FORA_INADIMPLENCIA_6M',
            'DIAS_ATRASO_MAX_6M',
            'DIAS_ATRASO_MIN_6M',
            'DIAS_ATRASO_MEDIA_6M',
            'QT_ATRASO_MAX_6M',
            'QT_ATRASO_MIN_6M',
            'QT_ATRASO_MEDIA_6M',
            'VALOR_COBRANCA_DIA_MEDIA_12M',
            'PRAZO_EMISSAO_VENCIMENTO_MAX_12M',
            'PRAZO_EMISSAO_VENCIMENTO_MIN_12M',
            'DIAS_EMISSAO_PAGAMENTO_MEDIA_12M',
            'PRAZO_EMISSAO_VENCIMENTO_MEDIA_12M',
            'VALOR_A_PAGAR_MEDIA_12M',
            'PCT_FORA_INADIMPLENCIA_12M',
            'PCT_EM_DIA_12M',
            'VALOR_A_PAGAR_MAX_12M',
            'VALOR_A_PAGAR_MIN_12M',
            'DIAS_ATRASO_MAX_12M',
            'DIAS_ATRASO_MIN_12M',
            'DIAS_ATRASO_MEDIA_12M',
            'QT_ATRASO_MAX_12M',
            'QT_ATRASO_MIN_12M',
            'QT_ATRASO_MEDIA_12M',
            'VALOR_COBRANCA_DIA_MEDIA_VIDA'
        ]
        self.cols_mediana = cols_restantes + [
            'RAZAO_RENDA_POR_FUNCIONARIO',
            'RAZAO_TAXA_VALOR_A_PAGAR',
            'DIF_PARA_MEDIA_PORTE'
        ]
        self.medianas_ = {}
        for col in self.cols_mediana:
            if col in X.columns:
                med = X[col].median()
                if pd.isna(med):
                    med = 0
                self.medianas_[col] = med

        return self

    def transform(self, X):
        X = X.copy()
        X["DDD"] = pd.to_numeric(X["DDD"], errors="coerce")
        X["PORTE"] = X["PORTE"].fillna("DESCONHECIDO")
        X["SEGMENTO_INDUSTRIAL"] = X["SEGMENTO_INDUSTRIAL"].fillna("desconhecido")
        X["DOMINIO_EMAIL"] = X["DOMINIO_EMAIL"].fillna("DESCONHECIDO")
        X["DDD"] = X["DDD"].mask(
            X["DDD"].isna(),
            X["ESTADO"].map(self.ddd_estado_)
        )
        X["DDD"] = X["DDD"].fillna(self.ddd_moda_)
        X["ESTADO"] = X["ESTADO"].fillna(self.estado_moda_)
        X["CEP_2_DIG"] = X["CEP_2_DIG"].fillna(self.cep_moda_)
        X["REGIAO"] = X["REGIAO"].fillna(self.regiao_moda_)

        # Corrige FLAG_SEM_HISTORICO_FUNCIONARIOS
        X["FLAG_SEM_HISTORICO_FUNCIONARIOS"] = (
            X["FLAG_PORTE_AUSENTE"].isna().astype("int8")
        )

        # Imputação por constantes
        for col in self.fill_zero_:
            if col in X.columns:
                X[col] = X[col].fillna(0)
        for col in self.fill_9999_:
            if col in X.columns:
                X[col] = X[col].fillna(9999)
        for col in self.fill_menos1_:
            if col in X.columns:
                X[col] = X[col].fillna(-1)

        # Imputação por média de porte e global
        for col in self.cols_nivel:
            X[f"FLAG_{col}_IMPUTADA"] = X[col].isna().astype("int8")
            medias = X["PORTE"].map(self.media_porte_[col])
            X[col] = (
                X[col]
                .fillna(medias)
                .fillna(self.media_global_[col])
            )

        # Flags e imputação para variáveis de variação e std
        # Optimize fragmentation: collect all flag columns, then concat once
        flag_variacao = {}
        for col in self.cols_variacao:
            if col in X.columns:
                flag_variacao[f"FLAG_{col}_AUSENTE"] = X[col].isna().astype("int8")
                X[col] = X[col].fillna(0)
        flag_std = {}
        for col in self.cols_std:
            if col in X.columns:
                flag_std[f"FLAG_{col}_AUSENTE"] = X[col].isna().astype("int8")
                X[col] = X[col].fillna(0)
        if flag_variacao or flag_std:
            X = pd.concat([X, pd.DataFrame({**flag_variacao, **flag_std}, index=X.index)], axis=1)

        # Imputação para MESES_CONSECUTIVOS_QUEDA
        if "MESES_CONSECUTIVOS_QUEDA" in X.columns:
            X["FLAG_SEM_HISTORICO_RENDA"] = (
                X["MESES_CONSECUTIVOS_QUEDA"]
                .isna()
                .astype("int8")
            )
            X["MESES_CONSECUTIVOS_QUEDA"] = (
                X["MESES_CONSECUTIVOS_QUEDA"]
                .fillna(0)
            )

        # Imputação por cascata
        flag_cascata = {}
        for _, cols in self.cascatas.items():
            principal = cols[0]
            flag_cascata[f"FLAG_{principal}_IMPUTADA_CASCATA"] = (
                X[principal].isna().astype("int8")
            )
            resultado = X[cols].bfill(axis=1).iloc[:,0]
            X[principal] = resultado.fillna(
                self.medianas_cascata_[principal]
            )
        if flag_cascata:
            X = pd.concat([X, pd.DataFrame(flag_cascata, index=X.index)], axis=1)

        # Imputação por mediana
        for col, med in self.medianas_.items():
            if col in X.columns:
                X[col] = X[col].fillna(med)
        return X

# Extração de features de data
def add_date_features(X):
    X = X.copy()
    X['ANO_REF'] = pd.to_datetime(X['DATA_REF']).dt.year
    X['MES_REF'] = pd.to_datetime(X['DATA_REF']).dt.month
    X['ANO_CADASTRO'] = pd.to_datetime(X['DATA_CADASTRO']).dt.year
    X['MES_CADASTRO'] = pd.to_datetime(X['DATA_CADASTRO']).dt.month
    X = X.drop(columns=['DATA_REF', 'DATA_CADASTRO'])
    return X

# Conversão ordinal para PORTE
def add_porte_ordinal(X):
    X = X.copy()
    porte_ordem = ['PEQUENO', 'MEDIO', 'GRANDE', 'DESCONHECIDO']
    X['PORTE'] = X['PORTE'].map({v: i for i, v in enumerate(porte_ordem)})
    return X

# Label encoding customizado para ESTADO e DDD
class LabelEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.estado_map_ = None
        self.ddd_map_ = None

    def fit(self, X, y=None):
        X = X.copy()
        self.estado_map_ = {
            v: i
            for i, v in enumerate(sorted(X["ESTADO"].astype(str).unique()))
        }
        self.ddd_map_ = {
            v: i
            for i, v in enumerate(sorted(X["DDD"].astype(str).unique()))
        }
        return self

    def transform(self, X):
        X = X.copy()
        X["ESTADO_LABEL"] = (
            X["ESTADO"]
            .astype(str)
            .map(self.estado_map_)
            .fillna(-1)
            .astype(int)
        )
        X["DDD_LABEL"] = (
            X["DDD"]
            .astype(str)
            .map(self.ddd_map_)
            .fillna(-1)
            .astype(int)
        )
        X = X.drop(columns=["ESTADO", "DDD"])
        return X

# Pré-processamento para definição de colunas numéricas
tmp = NullImputer().fit_transform(X_train)
tmp = add_date_features(tmp)
tmp = add_porte_ordinal(tmp)
tmp = LabelEncoderTransformer().fit_transform(tmp)
num_cols = tmp.select_dtypes(include=np.number).columns.tolist()
del tmp


# Modificação

Nesta etapa, é definido o **pipeline de pré-processamento**, que reúne todas as transformações necessárias em uma única sequência.

O pipeline realiza o **tratamento dos valores nulos**, a **extração das variáveis de data**, a **transformação de PORTE** e a **codificação de ESTADO e DDD**. Em seguida, as variáveis categóricas são transformadas com **One-Hot Encoding**, enquanto as variáveis numéricas são **normalizadas com MinMaxScaler**.

Dessa forma, todas as transformações são aplicadas de maneira padronizada aos dados de treino, validação e teste, evitando diferenças no pré-processamento entre os conjuntos.


In [0]:
# Pipeline de pré-processamento
preprocess_pipeline = Pipeline([
    ("null_imputer", NullImputer()),
    ("date_features", FunctionTransformer(add_date_features, validate=False)),
    ("porte_ordinal", FunctionTransformer(add_porte_ordinal, validate=False)),
    ("label_encodings", LabelEncoderTransformer()),
    ("column_transform", ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["SEGMENTO_INDUSTRIAL", "REGIAO", "DOMINIO_EMAIL"]),
            ("num", MinMaxScaler(), num_cols)
        ],
        remainder="passthrough"
    ))
])

# Modelagem

Foram testados três algoritmos de classificação: **XGBoost, Regressão Logística e Random Forest**. Para cada modelo, foi utilizado o mesmo pipeline de pré-processamento, garantindo uma comparação justa.

Os modelos foram treinados no conjunto de treino e avaliados nos conjuntos de **treino, validação e teste**. Como métricas principais, foram utilizadas **AUC-ROC e KS**, permitindo avaliar tanto a capacidade de separação entre adimplentes e inadimplentes quanto a estabilidade do modelo entre os diferentes conjuntos.

Além disso, os resultados de cada experimento foram registrados no **MLflow**, facilitando a comparação e o acompanhamento dos modelos.

**Modelos avaliados:**

* **XGBoost:** modelo baseado em árvores de decisão com boosting.
* **Regressão Logística:** modelo linear utilizado como referência e pela maior interpretabilidade.
* **Random Forest:** conjunto de árvores de decisão capaz de capturar relações não lineares.

Ao final, o modelo com melhor desempenho e maior capacidade de generalização foi selecionado para as etapas seguintes.


## Xgboost

In [0]:
# Define experimento MLflow
mlflow.set_experiment(experiment_id=3284700244160249)

with mlflow.start_run() as run:

    # Pipeline: pré-processamento + modelo XGBoost
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("xgb_model", XGBClassifier(eval_metric="logloss"))
    ])

    # Treina pipeline
    final_pipeline.fit(X_train, y_train)

    # Predições para cálculo de AUC
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Calcula AUC para cada conjunto
    auc_train = roc_auc_score(y_train, y_train_pred)
    auc_val   = roc_auc_score(y_val, y_val_pred)
    auc_test  = roc_auc_score(y_test, y_test_pred)

    # Calcula KS para cada conjunto
    ks_train = ks_2samp(y_train_pred[y_train == 1], y_train_pred[y_train == 0]).statistic
    ks_val   = ks_2samp(y_val_pred[y_val == 1], y_val_pred[y_val == 0]).statistic
    ks_test  = ks_2samp(y_test_pred[y_test == 1], y_test_pred[y_test == 0]).statistic

    # Loga métricas no MLflow
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_val", auc_val)
    mlflow.log_metric("auc_test", auc_test)
    mlflow.log_metric("ks_train", ks_train)
    mlflow.log_metric("ks_val", ks_val)
    mlflow.log_metric("ks_test", ks_test)

    print(f"AUC train: {auc_train:.4f}, val: {auc_val:.4f}, test: {auc_test:.4f}")
    print(f"KS train: {ks_train:.4f}, val: {ks_val:.4f}, test: {ks_test:.4f}")

## Logistic Regression

In [0]:
# Define experimento MLflow
mlflow.set_experiment(experiment_id=3284700244160249)

with mlflow.start_run() as run:

    # Pipeline: pré-processamento + modelo Logistic Regression
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("logreg_model", LogisticRegression(max_iter=1000))
    ])

    # Treina pipeline
    final_pipeline.fit(X_train, y_train)

    # Predições para cálculo de AUC
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Calcula AUC para cada conjunto
    auc_train = roc_auc_score(y_train, y_train_pred)
    auc_val   = roc_auc_score(y_val, y_val_pred)
    auc_test  = roc_auc_score(y_test, y_test_pred)

    # Calcula KS para cada conjunto
    ks_train = ks_2samp(y_train_pred[y_train == 1], y_train_pred[y_train == 0]).statistic
    ks_val   = ks_2samp(y_val_pred[y_val == 1], y_val_pred[y_val == 0]).statistic
    ks_test  = ks_2samp(y_test_pred[y_test == 1], y_test_pred[y_test == 0]).statistic

    # Loga métricas no MLflow
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_val", auc_val)
    mlflow.log_metric("auc_test", auc_test)
    mlflow.log_metric("ks_train", ks_train)
    mlflow.log_metric("ks_val", ks_val)
    mlflow.log_metric("ks_test", ks_test)

    print(f"AUC train: {auc_train:.4f}, val: {auc_val:.4f}, test: {auc_test:.4f}")
    print(f"KS train: {ks_train:.4f}, val: {ks_val:.4f}, test: {ks_test:.4f}")

## RandomForest

In [0]:
# Define experimento MLflow
mlflow.set_experiment(experiment_id=3284700244160249)

with mlflow.start_run() as run:

    # Pipeline: pré-processamento + modelo Random Forest
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("rf_model", RandomForestClassifier(n_estimators=100, random_state=42))
    ])

    # Treina pipeline
    final_pipeline.fit(X_train, y_train)

    # Predições para cálculo de AUC
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Calcula AUC para cada conjunto
    auc_train = roc_auc_score(y_train, y_train_pred)
    auc_val   = roc_auc_score(y_val, y_val_pred)
    auc_test  = roc_auc_score(y_test, y_test_pred)

    # Calcula KS para cada conjunto
    ks_train = ks_2samp(y_train_pred[y_train == 1], y_train_pred[y_train == 0]).statistic
    ks_val   = ks_2samp(y_val_pred[y_val == 1], y_val_pred[y_val == 0]).statistic
    ks_test  = ks_2samp(y_test_pred[y_test == 1], y_test_pred[y_test == 0]).statistic

    # Loga métricas no MLflow
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_val", auc_val)
    mlflow.log_metric("auc_test", auc_test)
    mlflow.log_metric("ks_train", ks_train)
    mlflow.log_metric("ks_val", ks_val)
    mlflow.log_metric("ks_test", ks_test)

    print(f"AUC train: {auc_train:.4f}, val: {auc_val:.4f}, test: {auc_test:.4f}")
    print(f"KS train: {ks_train:.4f}, val: {ks_val:.4f}, test: {ks_test:.4f}")

# Modelo Final


Após a comparação entre os modelos, o **XGBoost** foi selecionado para o modelo final. O pipeline completo, incluindo o pré-processamento e o modelo, foi treinado novamente e registrado no **MLflow**.

O desempenho obtido foi:

| Métrica | Treino | Validação |  Teste |
| ------- | -----: | --------: | -----: |
| **AUC** | 0.9964 |    0.9759 | 0.9748 |
| **KS**  | 0.9437 |    0.8471 | 0.8497 |

Os resultados de validação e teste são bastante próximos, indicando **boa capacidade de generalização**. O modelo apresentou AUC de **0.9748** e KS de **0.8497** no conjunto de teste.

Por fim, o pipeline completo foi **salvo no MLflow**, permitindo seu versionamento e posterior utilização para realizar novas previsões.


## Xgboost Final

In [0]:
# Inicia run MLflow
with mlflow.start_run() as run:

    # Pipeline: pré-processamento + modelo XGBoost
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("xgb_model", XGBClassifier(
            eval_metric="logloss" 
        ))
    ])

    # Treina pipeline
    final_pipeline.fit(X_train, y_train)

    # Predições para cálculo de AUC
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Calcula KS para cada conjunto
    ks_train = ks_2samp(y_train_pred[y_train == 1], y_train_pred[y_train == 0]).statistic
    ks_val   = ks_2samp(y_val_pred[y_val == 1], y_val_pred[y_val == 0]).statistic
    ks_test  = ks_2samp(y_test_pred[y_test == 1], y_test_pred[y_test == 0]).statistic

    # Loga métricas no MLflow
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_val", auc_val)
    mlflow.log_metric("auc_test", auc_test)
    mlflow.log_metric("ks_train", ks_train)
    mlflow.log_metric("ks_val", ks_val)
    mlflow.log_metric("ks_test", ks_test)
    
    # Salva modelo no MLflow
    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        name="model",
        input_example=X_train.iloc[:5]
    )

    print(run.info.run_id)  # Exibe run_id do MLflow

# Avaliação

## Chamando o Modelo e Prevendo Teste e Validação

In [0]:
model = mlflow.sklearn.load_model("runs:/e0d38d0f7d3a4d2a957ef79069735096/model")

In [0]:
# Cria DataFrames de teste e validação com target
df_test = X_test.copy()
df_test["target"] = y_test

df_val = X_val.copy()
df_val["target"] = y_val

# Gera predições e probabilidades do modelo
df_val["pred"] = model.predict(df_val)
df_val["proba"] = model.predict_proba(df_val)[:, 1]
df_test["pred"] = model.predict(df_test)
df_test["proba"] = model.predict_proba(df_test)[:, 1]

## Taxa de Inadimplência por Score do Modelo


Esta análise foi realizada para **verificar se o score gerado pelo modelo realmente representa diferentes níveis de risco de crédito**. Para isso, os clientes são separados em faixas de probabilidade e, em seguida, é calculada a taxa de inadimplência observada em cada faixa.

Primeiramente, as probabilidades geradas pelo modelo são divididas em intervalos de risco:

| Faixa de score | Interpretação |
| -------------- | ------------- |
| 0,00 – 0,10    | Baixo risco   |
| 0,10 – 0,20    | Risco         |
| 0,20 – 0,30    | Risco         |
| 0,30 – 0,40    | Risco         |
| 0,40 – 0,50    | Risco         |
| 0,50 – 0,60    | Alto risco    |
| 0,60 – 1,00    | Alto risco    |

Depois, para cada faixa, são contabilizados:

* **Total de clientes:** quantidade de clientes naquela faixa;
* **Inadimplentes:** quantidade de clientes inadimplentes;
* **Taxa de inadimplência:** proporção de inadimplentes dentro da faixa.

A mesma análise é realizada separadamente nos conjuntos de **validação e teste**. Isso permite verificar se o comportamento observado no treinamento/validação se mantém em dados que o modelo não utilizou para seu ajuste.

Em seguida, as taxas de inadimplência dos conjuntos de validação e teste são agrupadas e é calculada uma **média entre os dois conjuntos**, permitindo obter uma visão geral do comportamento do modelo.

Por fim, é gerado um gráfico de barras comparando a taxa de inadimplência de cada faixa entre validação e teste.

O principal objetivo é verificar se existe uma **relação crescente entre o score do modelo e a inadimplência observada**. Um modelo bem calibrado ou, pelo menos, bem ordenado em termos de risco deve apresentar, de maneira geral, **menores taxas de inadimplência nas faixas de menor score e maiores taxas nas faixas de maior score**.


In [0]:
# Define faixas de risco para o score do modelo
bins = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 1.0]
labels = [f"{bins[i]:.2f}-{bins[i+1]:.2f}" for i in range(len(bins)-1)]

df_val['faixa_risco'] = pd.cut(df_val['proba'], bins=bins, labels=labels, include_lowest=True, right=False)
df_test['faixa_risco'] = pd.cut(df_test['proba'], bins=bins, labels=labels, include_lowest=True, right=False)

In [0]:
# Agrupa por faixa_risco e calcula porcentagem de inadimplentes para validação
df_val_grouped = df_val.groupby('faixa_risco', observed=False).agg(
    total_val=('target', 'count'),
    inadimplentes_val=('target', 'sum')
)
df_val_grouped['porcentagem_inadimplentes_val'] = 100 * df_val_grouped['inadimplentes_val'] / df_val_grouped['total_val']

# Agrupa por faixa_risco e calcula porcentagem de inadimplentes para teste
df_test_grouped = df_test.groupby('faixa_risco', observed=False).agg(
    total_test=('target', 'count'),
    inadimplentes_test=('target', 'sum')
)
df_test_grouped['porcentagem_inadimplentes_test'] = 100 * df_test_grouped['inadimplentes_test'] / df_test_grouped['total_test']

# Junta as porcentagens de inadimplentes de validação e teste
df_grouped = df_val_grouped[['porcentagem_inadimplentes_val']].join(
    df_test_grouped[['porcentagem_inadimplentes_test']],
    how='outer'
)

# Calcula a média das porcentagens de inadimplentes
df_grouped['media_porcentagem_inadimplentes'] = df_grouped[
    ['porcentagem_inadimplentes_val', 'porcentagem_inadimplentes_test']
].mean(axis=1)

# Formata a média como porcentagem inteira
df_grouped['media_porcentagem_inadimplentes'] = df_grouped['media_porcentagem_inadimplentes'].apply(lambda x: f"{x:.0f}%")

# Mantém apenas faixa_risco e média para visualização
df_grouped = df_grouped[['media_porcentagem_inadimplentes']].reset_index()

display(df_grouped)

In [0]:
import plotly.graph_objects as go

# Junta as porcentagens de inadimplentes de validação e teste
df_hist = df_val_grouped[['porcentagem_inadimplentes_val']].join(
    df_test_grouped[['porcentagem_inadimplentes_test']],
    how='outer'
)

labels = df_hist.index.astype(str)

fig = go.Figure(data=[
    go.Bar(
        name='Validação',
        x=labels,
        y=df_hist['porcentagem_inadimplentes_val'],
        marker_color='rgb(55,55,55)',
        opacity=0.55
    ),
    go.Bar(
        name='Teste',
        x=labels,
        y=df_hist['porcentagem_inadimplentes_test'],
        marker_color='black',
        opacity=1.0
    )
])

fig.update_layout(
    barmode='group',
    xaxis_title='Faixa de Risco',
    yaxis_title='Porcentagem de Inadimplência',
    title='Porcentagem de Inadimplência por Faixa de Risco',
    legend_title=None,
    xaxis_tickangle=-45,
    template='simple_white',
    width=1000,
    height=600
)

fig.show()

# Analise Financeira

## Analise de Treshold


Esta etapa tem como objetivo **definir o melhor ponto de corte para transformar a probabilidade de inadimplência gerada pelo modelo em uma classificação binária**.

Foram testados thresholds entre **0 e 1**, com intervalos de 0,01, utilizando o conjunto de validação. Para cada threshold, foi calculado o **F1 Score**, buscando um equilíbrio entre precisão e recall.

O threshold com maior F1 Score foi selecionado e posteriormente aplicado **sem alterações ao conjunto de teste**, evitando utilizar o teste na escolha do modelo.

No conjunto de teste, foram calculadas **acurácia, precisão, recall, F1 Score e AUC-ROC**, além da matriz de confusão, permitindo avaliar tanto o desempenho geral quanto a capacidade do modelo de identificar corretamente os clientes inadimplentes.


In [0]:
# Busca o melhor threshold para maximizar o F1 score no conjunto de validação
y_val_proba = df_val['proba']
best_threshold = 0.5
best_f1 = 0

thresholds = np.arange(0.0, 1.01, 0.01)
f1_scores = []

for threshold in thresholds:
    y_pred = (y_val_proba >= threshold).astype(int)
    score = f1_score(y_val, y_pred)
    f1_scores.append(score)
    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print(f"Melhor threshold para F1 no conjunto de validação: {best_threshold:.2f} (F1={best_f1:.4f})")

# Visualiza a curva F1 x Threshold
plt.figure(figsize=(8,5))
plt.plot(thresholds, f1_scores, marker='o')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score por Threshold no conjunto de validação')
plt.grid(True)
plt.show()

In [0]:
# Gera predições binárias com threshold 0.34 para o conjunto de teste
y_test_proba = df_test['proba']
threshold = 0.34
y_test_pred_t34 = (y_test_proba >= threshold).astype(int)

# Matriz de confusão para threshold 0.34
cm = confusion_matrix(y_test, y_test_pred_t34)
print("Matriz de confusão para threshold 0.34 no conjunto de teste:")
print(cm)

# Calcula métricas de avaliação
acc = accuracy_score(y_test, y_test_pred_t34)
prec = precision_score(y_test, y_test_pred_t34)
rec = recall_score(y_test, y_test_pred_t34)
f1 = f1_score(y_test, y_test_pred_t34)
auc = roc_auc_score(y_test, y_test_proba)  # AUC usa as probabilidades

print(f"\nAcurácia:  {acc:.4f}")
print(f"Precisão:  {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"AUC:       {auc:.4f}")

## Política de Crédito VS Modelo

Esta etapa tem como objetivo **comparar o impacto financeiro do modelo de crédito com uma política de crédito utilizada como proxy**.

Primeiramente, o valor a pagar de cada documento é recuperado do banco de dados e associado ao conjunto de teste. Em seguida, são realizadas as decisões de crédito utilizando duas abordagens: a **proxy**, baseada no histórico de inadimplência do cliente, e o **modelo**, baseado na probabilidade de inadimplência.

Para tornar a comparação justa, o modelo é ajustado para **negar o mesmo percentual de documentos que a proxy**. Dessa forma, ambas as estratégias trabalham com aproximadamente a mesma taxa de aprovação, permitindo avaliar qual delas seleciona uma carteira de melhor qualidade.

São então comparados:

* **Taxa de aprovação:** percentual de clientes aprovados;
* **Inadimplência dos aprovados:** percentual de clientes inadimplentes dentro da carteira aprovada;
* **Loss Rate:** proporção do valor aprovado que resulta em perda;
* **Valor aprovado:** soma dos valores dos documentos aprovados;
* **Valor perdido:** valor associado aos clientes inadimplentes aprovados;
* **Resultado financeiro:** valor dos clientes adimplentes aprovados menos o valor dos inadimplentes aprovados.

Por fim, são calculadas as diferenças entre o modelo e a proxy, permitindo verificar **quanto o modelo gera a mais e quanto consegue reduzir as perdas financeiras**, mantendo o mesmo nível de aprovação da política utilizada como referência.


In [0]:
# Busca valores a pagar por documento no banco de dados
valor_a_pagar_df = spark.sql("""
    SELECT ID_DOCUMENTO, VALOR_A_PAGAR
    FROM credit_score.data.pagamentos
""").toPandas()

# Adiciona coluna VALOR_A_PAGAR ao df_test
df_test = df_test.merge(valor_a_pagar_df, on="ID_DOCUMENTO", how="left")

In [0]:
# Predição da proxy: aprova se PCT_FORA_INADIMPLENCIA_12M < 0.8
df_test['proxy_pred'] = (df_test['PCT_FORA_INADIMPLENCIA_12M'] < 0.8).astype(int)
# Predição do modelo: aprova se proba >= 0.25
df_test['pred'] = (df_test['proba'] >= 0.25).astype(int)

# Percentual de documentos negados pela proxy
percentual_negados_proxy = (df_test['proxy_pred'] == 1).mean()
print(f"Percentual de documentos negados pela proxy: {percentual_negados_proxy:.2%}")

# Valor total ganho pela proxy (aprovados adimplentes menos aprovados inadimplentes)
valor_ganho_aprovados_adimplentes_proxy = df_test.loc[(df_test['proxy_pred'] == 0) & (df_test['target'] == 0), 'VALOR_A_PAGAR'].sum()
valor_perdido_aprovados_inadimplentes_proxy = df_test.loc[(df_test['proxy_pred'] == 0) & (df_test['target'] == 1), 'VALOR_A_PAGAR'].sum()
valor_total_ganho_proxy = valor_ganho_aprovados_adimplentes_proxy - valor_perdido_aprovados_inadimplentes_proxy
print(f"Proxy - Valor total ganho: {valor_total_ganho_proxy:.0f}")

# Valor perdido pela proxy (adimplentes negados)
valor_perdido_proxy = df_test.loc[(df_test['proxy_pred'] == 1) & (df_test['target'] == 0), 'VALOR_A_PAGAR'].sum()
print(f"Proxy - Valor perdido: {valor_perdido_proxy:.0f}")

# Define ponto de corte do modelo para negar o mesmo percentual de linhas que a proxy
threshold_model = df_test['proba'].quantile(1 - percentual_negados_proxy)
df_test['pred_equal_proxy'] = (df_test['proba'] >= threshold_model).astype(int)

# Valor total ganho pelo modelo (aprovados adimplentes menos aprovados inadimplentes)
valor_ganho_aprovados_adimplentes_modelo_eq = df_test.loc[(df_test['pred_equal_proxy'] == 0) & (df_test['target'] == 0), 'VALOR_A_PAGAR'].sum()
valor_perdido_aprovados_inadimplentes_modelo_eq = df_test.loc[(df_test['pred_equal_proxy'] == 0) & (df_test['target'] == 1), 'VALOR_A_PAGAR'].sum()
valor_total_ganho_modelo_eq = valor_ganho_aprovados_adimplentes_modelo_eq - valor_perdido_aprovados_inadimplentes_modelo_eq
print(f"Modelo - Valor total ganho: {valor_total_ganho_modelo_eq:.0f}")

# Valor perdido pelo modelo (adimplentes negados)
valor_perdido_modelo_eq = df_test.loc[(df_test['pred_equal_proxy'] == 1) & (df_test['target'] == 0), 'VALOR_A_PAGAR'].sum()
print(f"Modelo - Valor perdido: {valor_perdido_modelo_eq:.0f}")

# Comparação: quanto o modelo gera a mais que a proxy com ponto de corte igual
valor_gerado_a_mais_eq = valor_total_ganho_modelo_eq - valor_total_ganho_proxy
print(f"\nComparação: O modelo gera a mais que a proxy: {valor_gerado_a_mais_eq:.0f}")

# Comparação: quanto o modelo deixa de ganhar a menos que a proxy
reducao_perdas =  valor_perdido_proxy - valor_perdido_modelo_eq
print(f"Comparação: O modelo tem uma redução de perdas em relação à proxy de: {reducao_perdas:.0f}")


In [0]:
# Calcula inadimplência dos aprovados pela proxy e pelo modelo
inadimplencia_aprovados_proxy = df_test.loc[df_test['proxy_pred'] == 0, 'target'].mean()
inadimplencia_aprovados_modelo = df_test.loc[df_test['pred_equal_proxy'] == 0, 'target'].mean()

# Soma dos valores aprovados pela proxy e pelo modelo
valor_aprovado_proxy = df_test.loc[df_test['proxy_pred'] == 0, 'VALOR_A_PAGAR'].sum()
valor_aprovado_modelo = df_test.loc[df_test['pred_equal_proxy'] == 0, 'VALOR_A_PAGAR'].sum()

# Calcula loss rate (percentual perdido sobre valor aprovado)
loss_rate_proxy = valor_perdido_aprovados_inadimplentes_proxy / valor_aprovado_proxy
loss_rate_modelo = valor_perdido_aprovados_inadimplentes_modelo_eq / valor_aprovado_modelo

# Percentual de aprovação pela proxy e pelo modelo
aprovacao_proxy = (df_test['proxy_pred'] == 0).mean()
aprovacao_modelo = (df_test['pred_equal_proxy'] == 0).mean()

# Percentual do valor gerado a mais pelo modelo em relação à proxy
percentual_valor_gerado_a_mais_eq = valor_gerado_a_mais_eq / abs(valor_total_ganho_proxy) * 100

# Percentual do valor perdido a menos pelo modelo em relação à proxy
pct_reducao_perdas = reducao_perdas / abs(valor_perdido_proxy) * 100

In [0]:
print(f"Aprovação proxy: {aprovacao_proxy:.2%}")
print(f"Aprovação modelo: {aprovacao_modelo:.2%}\n")

print(f"Inadimplência dos aprovados pela proxy: {inadimplencia_aprovados_proxy:.2%}")
print(f"Inadimplência dos aprovados pelo modelo: {inadimplencia_aprovados_modelo:.2%}\n")

print(f"Loss rate proxy: {loss_rate_proxy:.2%}")
print(f"Loss rate modelo: {loss_rate_modelo:.2%}\n")

print(f"Valor aprovado proxy: {valor_aprovado_proxy:.0f}")
print(f"Valor aprovado modelo: {valor_aprovado_modelo:.0f}\n")

print(f"Valor perdido proxy: {valor_perdido_proxy:.0f}")
print(f"Valor perdido modelo: {valor_perdido_modelo_eq:.0f}\n")

print(f"Percentual de Valor gerado a mais pelo modelo: {percentual_valor_gerado_a_mais_eq:.0f}%")
print(f"Percentual de Reducao de perdas do modelo: {pct_reducao_perdas:.0f}%")

# Conclusão


O modelo apresentou **boa capacidade de separar clientes de diferentes níveis de risco**. As faixas de probabilidade mostram uma relação clara entre o risco previsto pelo modelo e a inadimplência observada:

| Faixa de risco | Inadimplência |
| -------------- | ------------: |
| 0,00 – 0,10    |        **1%** |
| 0,10 – 0,20    |       **24%** |
| 0,20 – 0,30    |       **34%** |
| 0,30 – 0,40    |       **40%** |
| 0,40 – 0,50    |       **50%** |
| 0,50 – 0,60    |       **56%** |
| 0,60 – 1,00    |       **79%** |

Isso indica que, conforme a probabilidade prevista aumenta, a taxa de inadimplência também aumenta, mostrando uma **boa ordenação dos clientes por risco**.

### Desempenho do modelo

O melhor threshold encontrado na validação foi **0,34**, com F1 Score de **0,7024**.

| Métrica  |      Teste |
| -------- | ---------: |
| AUC-ROC  | **0,9748** |
| Acurácia | **95,74%** |
| Precisão | **68,62%** |
| Recall   | **65,52%** |
| F1 Score | **0,6703** |

### Impacto financeiro

Para comparar o modelo de forma justa com a política de crédito utilizada como proxy, ambos foram avaliados com a **mesma taxa de aprovação: 90,23%**.

| Indicador                   |       Proxy |          Modelo |
| --------------------------- | ----------: | --------------: |
| Aprovação                   |      90,23% |      **90,23%** |
| Inadimplência dos aprovados |       2,64% |       **1,23%** |
| Loss Rate                   |       1,99% |       **1,20%** |
| Valor aprovado              | R$ 1,346 bi | **R$ 1,359 bi** |
| Valor perdido               |  R$ 67,1 mi |  **R$ 43,9 mi** |
| Resultado financeiro        | R$ 1,293 bi | **R$ 1,327 bi** |

O modelo, portanto, apresentou:

* **53% menos inadimplência entre os aprovados**;
* **34% de redução nas perdas financeiras**;
* aproximadamente **R$ 23,1 milhões a menos em perdas**;
* aproximadamente **R$ 33,7 milhões de resultado adicional**;
* **3% de aumento no valor gerado**.

### Conclusão final

De forma geral, os resultados mostram que o modelo consegue **manter a mesma taxa de aprovação da política atual e, ao mesmo tempo, reduzir significativamente o risco da carteira**. Além do bom desempenho estatístico, a análise financeira indica um potencial de melhoria relevante na decisão de crédito, principalmente pela redução das perdas e da inadimplência entre os clientes aprovados.

> **Em resumo: o modelo consegue aprovar a mesma quantidade de clientes, porém selecionando uma carteira com menor risco e menor perda financeira.**


##